# Проверка масштабируемости рекомендательной системы

Основная разработка рекомендательной системы проводилась на выборке из 10 000 пользовательских сессий. Это позволило быстрее проводить эксперименты с генерацией кандидатов, признаками и моделью ранжирования.

На данном этапе проверим работу разработанного подхода на большей выборке из 100 000 сессий.

Цели эксперимента:

- проверить, способен ли построенный pipeline работать на большем объёме данных;
- оценить изменение времени выполнения основных этапов;
- проверить, сохраняется ли качество рекомендательной системы при увеличении выборки;
- сравнить модель ранжирования с baseline на одинаковых validation-сессиях.

## Загрузка данных

Для эксперимента используем первые 100 000 сессий исходного обучающего датасета.

In [38]:
import pandas as pd
import numpy as np
from pathlib import Path
import time

data_path = Path("../data/otto-recsys-train.jsonl")

n_sessions = 100_000

start_time = time.time()

df = pd.read_json(
    data_path,
    lines=True,
    nrows=n_sessions
)

print("Количество сессий:", len(df))

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.
Количество сессий: 100000


## Разделение сессий на history и future

Разделяем сессий по знакомой схеме: 75% событий как историю пользователей (`history`'), а оставшиеся 25% - как будущие события для проверки работы модели (`future`).

In [39]:
def split_session(events):
    future_size = max(1, int(len(events) * 0.25))
    history = events[:-future_size]
    future = events[-future_size:]

    return history, future

start_time = time.time()

df[['history', 'future']] = df['events'].apply(
    lambda events: pd.Series(split_session(events))
)

split_time = time.time() - start_time
print(f"Время разделения: {split_time:.2f} сек.")
df[['session', 'history', 'future']].head()

Время разделения: 34.13 сек.


,session,history,future
0,0,"[{'aid': 1517085, 'ts': 1659304800025, 'type':...","[{'aid': 362233, 'ts': 1661551907312, 'type': ..."
1,1,"[{'aid': 424964, 'ts': 1659304800025, 'type': ...","[{'aid': 959544, 'ts': 1659991199405, 'type': ..."
2,2,"[{'aid': 763743, 'ts': 1659304800038, 'type': ...","[{'aid': 295985, 'ts': 1661713601647, 'type': ..."
3,3,"[{'aid': 1425967, 'ts': 1659304800095, 'type':...","[{'aid': 88711, 'ts': 1659998456613, 'type': '..."
4,4,"[{'aid': 613619, 'ts': 1659304800119, 'type': ...","[{'aid': 758750, 'ts': 1661504510200, 'type': ..."


## Генерация кандидатов

На основе `history` каждой сессии сформируем набор товаров-кандидатов, которые затем будут переданы модели ранжирования.

Используем ту же логику генерации кандидатов, что и в основном эксперименте:

1. товары из истории текущей сессии;
2. товары, найденные с помощью co-visitation;
3. популярные товары для дополнения пула.

In [6]:
from itertools import combinations

start_time = time.time()

pair_counts = {}

for history in df['history']:
    unique_items = set()
    
    for event in history:
        unique_items.add(event['aid'])

    for item_1, item_2 in combinations(unique_items, 2):
        pair = (item_1, item_2)

        if pair not in pair_counts:
            pair_counts[pair] = 0
        pair_counts[pair] += 1

covisitation_time = time.time() - start_time

print("Количество связей:", len(pair_counts))
print(f"Время построения связей: {covisitation_time:.2f} сек.")

Количество связей: 73717109
Время построения связей: 30.64 сек.


### Построение списка соседей товаров

In [7]:
start_time = time.time()

item_neighbors = {}
for (item_1, item_2), count in pair_counts.items():
    if item_1 not in item_neighbors:
        item_neighbors[item_1] = []
    if item_2 not in item_neighbors:
        item_neighbors[item_2] = []

    item_neighbors[item_1].append((item_2, count))
    item_neighbors[item_2].append((item_1, count))


def get_count(pair):
    return pair[1]

top_n = 20

for item in item_neighbors:
    item_neighbors[item] = sorted(
        item_neighbors[item],
        key=get_count,
        reverse=True
    )[:top_n]

neighbors_time = time.time() - start_time

print("Количество товаров:", len(item_neighbors))
print(f"Время построения соседей: {neighbors_time:.2f} сек.")

Количество товаров: 569788
Время построения соседей: 166.94 сек.


### Формирование пула кандидатов

Для каждой сессии формируем пул максимум из 100 товаров.

In [8]:
item_counts = {}

for history in df['history']:
    for event in history:
        aid = event['aid']

        if aid not in item_counts:
            item_counts[aid] = 0

        item_counts[aid] += 1

popular_items = sorted(
    item_counts,
    key=item_counts.get,
    reverse=True
)[:100]

print("Количество популярных товаров:", len(popular_items))
print(popular_items[:10])

Количество популярных товаров: 100
[29735, 832192, 1733943, 80222, 108125, 1083665, 166037, 554660, 1743151, 1603001]


In [9]:
def generate_candidates(history, item_neighbors, popular_items, max_candidates=100):
    candidates = []
    seen = set()

    for event in reversed(history):
        aid = event['aid']
        if aid not in seen:
            candidates.append(aid)
            seen.add(aid)
            
        if len(candidates) >= max_candidates:
            return candidates
            
    for event in reversed(history):
        aid = event['aid']
        for neighbor, count in item_neighbors.get(aid, []):
            if neighbor not in seen:
                candidates.append(neighbor)
                seen.add(neighbor)
                
            if len(candidates) >= max_candidates:
                return candidates

    for aid in popular_items:
        if aid not in seen:
            candidates.append(aid)
            seen.add(aid)

        if len(candidates) >= max_candidates:
            return candidates

    return candidates

### Генерация кандидатов для всех сессий

In [10]:
start_time = time.time()
candidate_pools = {}

for session_id, history in zip(df['session'], df['history']):
    candidate_pools[session_id] = generate_candidates(history,item_neighbors,popular_items)

candidate_generation_time = time.time() - start_time

print("Количество сессий:", len(candidate_pools))
print(f"Время генерации кандидатов: {candidate_generation_time:.2f} сек.")

Количество сессий: 100000
Время генерации кандидатов: 7.99 сек.


### Размер пула кандидатов

Проверим распределение количества кандидатов в сессиях после генерации. Максимальный размер пула ограничен 100 товарами.

In [11]:
candidate_sizes = []

for candidates in candidate_pools.values():
    candidate_sizes.append(len(candidates))

pd.Series(candidate_sizes).describe()

count    100000.0
mean        100.0
std           0.0
min         100.0
25%         100.0
50%         100.0
75%         100.0
max         100.0
dtype: float64

## Recall пула кандидатов

In [12]:
future_orders = {}

for session_id, future in zip(df['session'], df['future']):
    orders = set()

    for event in future:
        if event['type'] == 'orders':
            orders.add(event['aid'])

    future_orders[session_id] = orders

Посчитаем **Recall@100** для `orders`.

In [14]:
recalls = []

for session_id, relevant_items in future_orders.items():
    if len(relevant_items) == 0:
        continue
    candidates = set(candidate_pools[session_id])
    hits = len(relevant_items & candidates)
    recall = hits / len(relevant_items)

    recalls.append(recall)

print("Количество сессий с orders:", len(recalls))
print(f"Recall@100 orders: {sum(recalls) / len(recalls):.4f}")

Количество сессий с orders: 13094
Recall@100 orders: 0.5854


## Подготовка данных для модели ранжирования

Следующий этап — построить признаки для пар `(session, candidate)` и проверить, как увеличение объёма данных на обучении влияет на качество модели ранжирования.

## Построение признаков

Для каждого товара-кандидата построим признаки, описывающие его поведение на `history`.

In [15]:
start_time = time.time()

item_stats = {}

for history in df['history']:
    for event in history:
        aid = event['aid']
        event_type = event['type']

        if aid not in item_stats:
            item_stats[aid] = {
                'clicks': 0,
                'carts': 0,
                'orders': 0,
                'total': 0}

        item_stats[aid]['total'] += 1
        item_stats[aid][event_type] += 1

stats_time = time.time() - start_time
print("Количество товаров:", len(item_stats))
print(f"Время построения статистики: {stats_time:.2f} сек.")

Количество товаров: 571104
Время построения статистики: 2.29 сек.


### Временные признаки

Для каждого товара рассчитаем:

- `item_recency` — насколько давно товар последний раз встречался в истории конкретной сессии;
- `item_interaction_span` — временной промежуток между первым и последним взаимодействием с товаром в сессии.

In [16]:
for aid in item_stats:
    total = item_stats[aid]['total']

    item_stats[aid]['click_rate'] = item_stats[aid]['clicks'] / total
    item_stats[aid]['cart_rate'] = item_stats[aid]['carts'] / total
    item_stats[aid]['order_rate'] = item_stats[aid]['orders'] / total


### Формирование признаков для пары session–candidate

Для каждой пары `(session, candidate)` объединяем:

- глобальную статистику товара;
- характеристики взаимодействий с товаром внутри текущей сессии;
- силу co-visitation связи товара с историей сессии.

In [24]:
def get_candidate_features(candidate, history):
    candidate_events = []

    for event in history:
        if event['aid'] == candidate:
            candidate_events.append(event)

    if len(candidate_events) > 0:
        last_ts = history[-1]['ts']

        first_item_ts = candidate_events[0]['ts']
        last_item_ts = candidate_events[-1]['ts']

        item_recency = last_ts - last_item_ts
        item_interaction_span = last_item_ts - first_item_ts
    else:
        item_recency = -1
        item_interaction_span = 0






    
    
    stats = item_stats.get(candidate)

    if stats is None:
        click_count = 0
        cart_count = 0
        order_count = 0
        total_count = 0

        click_rate = 0
        cart_rate = 0
        order_rate = 0
    else:
        click_count = stats['clicks']
        cart_count = stats['carts']
        order_count = stats['orders']
        total_count = stats['total']

        click_rate = stats['click_rate']
        cart_rate = stats['cart_rate']
        order_rate = stats['order_rate']

    covisitation_score = get_covisitation_score(candidate, history)

    return {
        'item_click_count': click_count,
        'item_cart_count': cart_count,
        'item_order_count': order_count,
        'item_total_count': total_count,
        'item_recency': item_recency,
        'item_interaction_span': item_interaction_span,
        'covisitation_score': covisitation_score,
        'item_click_rate': click_rate,
        'item_cart_rate': cart_rate,
        'item_order_rate': order_rate
    }

### Сила co-visitation

Для каждого товара-кандидата рассчитаем силу его связи с товарами из истории текущей сессии.

In [25]:
def get_covisitation_score(candidate, history):
    history_items = set()

    for event in history:
        history_items.add(event['aid'])
    score = 0

    for history_item in history_items:
        for neighbor, count in item_neighbors.get(history_item, []):
            if neighbor == candidate:
                score += count
                break

    return score

### Проверка построения признаков

Перед формированием полного датасета проверим функцию построения признаков на одном товаре-кандидате из одной сессии.

In [26]:
test_session = df.iloc[0]

session_id = test_session['session']
history = test_session['history']

test_candidate = candidate_pools[session_id][0]

features = get_candidate_features(
    test_candidate,
    history
)

print("Session:", session_id)
print("Candidate:", test_candidate)
print()

for feature, value in features.items():
    print(feature, ":", value)

Session: 0
Candidate: 661144

item_click_count : 31
item_cart_count : 2
item_order_count : 1
item_total_count : 34
item_recency : 0
item_interaction_span : 172938592
covisitation_score : 15
item_click_rate : 0.9117647058823529
item_cart_rate : 0.058823529411764705
item_order_rate : 0.029411764705882353


### Оценка времени построения датасета

In [27]:
start_time = time.time()
test_rows = []

for _, row in df.iloc[:1000].iterrows():
    session_id = row['session']
    history = row['history']

    for candidate in candidate_pools[session_id]:
        features = get_candidate_features(candidate, history)

        features['session'] = session_id
        features['aid'] = candidate
        test_rows.append(features)

test_time = time.time() - start_time

print("Количество строк:", len(test_rows))
print(f"Время: {test_time:.2f} сек.")
print(f"Оценка для 100 000 сессий: {test_time * 100:.1f} сек.")

Количество строк: 100000
Время: 2.09 сек.
Оценка для 100 000 сессий: 209.0 сек.


## Формирование датасета для ранжирования

Для каждой пары `(session, candidate)` сформируем строку с признаками и целевой переменной:

- `0` — товар отсутствует в future;
- `1` — click;
- `2` — cart;
- `3` — order.

In [29]:
def get_target(candidate, future):
    target = 0

    for event in future:
        if event['aid'] != candidate:
            continue

        if event['type'] == 'clicks':
            target = max(target, 1)
        elif event['type'] == 'carts':
            target = max(target, 2)
        elif event['type'] == 'orders':
            target = max(target, 3)

    return target

### Проверка целевой переменной

In [30]:
test_session = df.iloc[0]

session_id = test_session['session']
future = test_session['future']

print("Session:", session_id)
print("Future events:", len(future))
print()

for candidate in candidate_pools[session_id][:10]:
    target = get_target(candidate, future)
    print("Candidate:", candidate, "| target:", target)

Session: 0
Future events: 69

Candidate: 661144 | target: 0
Candidate: 570955 | target: 0
Candidate: 1760145 | target: 1
Candidate: 974651 | target: 1
Candidate: 275288 | target: 1
Candidate: 543308 | target: 3
Candidate: 138431 | target: 0
Candidate: 1157411 | target: 0
Candidate: 190818 | target: 0
Candidate: 102416 | target: 0


### Подготовка сохранения датасета

Разделяем выборку на батчи по 5000 сессий.

In [33]:
from pathlib import Path

SCALING_PATH = Path("../artifacts/scaling_dataset")
SCALING_PATH.mkdir(parents=True, exist_ok=True)

batch_size = 5_000

### Формирование датасета батчами

Разбиваем 100 000 сессий на батчи по 5 000 сессий.

Для каждой пары `(session, candidate)`:
1. строим признаки;
2. определяем `target` по будущей части сессии;
3. добавляем идентификаторы сессии и товара;
4. сохраняем полученный батч на диск.

После сохранения очередного батча данные удаляются из оперативной памяти.

In [34]:
import gc

start_time = time.time()

for batch_num, start_idx in enumerate(range(0, len(df), batch_size)):
    end_idx = min(start_idx + batch_size, len(df))

    batch_rows = []

    for _, row in df.iloc[start_idx:end_idx].iterrows():
        session_id = row['session']
        history = row['history']
        future = row['future']

        for candidate in candidate_pools[session_id]:
            features = get_candidate_features(candidate, history)

            features['session'] = session_id
            features['aid'] = candidate
            features['target'] = get_target(candidate, future)

            batch_rows.append(features)

    batch_df = pd.DataFrame(batch_rows)

    file_path = SCALING_PATH / f"batch_{batch_num:02d}.pkl"
    batch_df.to_pickle(file_path)

    print(
        f"Батч {batch_num + 1}: "
        f"сессии {start_idx}-{end_idx - 1}, "
        f"строк {len(batch_df):,}"
    )

    del batch_rows
    del batch_df
    gc.collect()

total_time = time.time() - start_time

print()
print(f"Общее время: {total_time:.2f} сек.")

Батч 1: сессии 0-4999, строк 500,000
Батч 2: сессии 5000-9999, строк 500,000
Батч 3: сессии 10000-14999, строк 500,000
Батч 4: сессии 15000-19999, строк 500,000
Батч 5: сессии 20000-24999, строк 500,000
Батч 6: сессии 25000-29999, строк 500,000
Батч 7: сессии 30000-34999, строк 500,000
Батч 8: сессии 35000-39999, строк 500,000
Батч 9: сессии 40000-44999, строк 500,000
Батч 10: сессии 45000-49999, строк 500,000
Батч 11: сессии 50000-54999, строк 500,000
Батч 12: сессии 55000-59999, строк 500,000
Батч 13: сессии 60000-64999, строк 500,000
Батч 14: сессии 65000-69999, строк 500,000
Батч 15: сессии 70000-74999, строк 500,000
Батч 16: сессии 75000-79999, строк 500,000
Батч 17: сессии 80000-84999, строк 500,000
Батч 18: сессии 85000-89999, строк 500,000
Батч 19: сессии 90000-94999, строк 500,000
Батч 20: сессии 95000-99999, строк 500,000

Общее время: 300.57 сек.


## Проверка сформированного датасета


Проверим:
- количество сохранённых батчей;
- общее количество строк;
- распределение целевой переменной;
- наличие пропущенных значений.

In [35]:
batch_files = sorted(SCALING_PATH.glob("batch_*.pkl"))

print("Количество батчей:", len(batch_files))

total_rows = 0
target_counts = pd.Series(dtype="int64")
missing_counts = None

for file_path in batch_files:
    batch = pd.read_pickle(file_path)

    total_rows += len(batch)

    target_counts = target_counts.add(
        batch["target"].value_counts(),
        fill_value=0
    )

    if missing_counts is None:
        missing_counts = batch.isna().sum()
    else:
        missing_counts = missing_counts.add(
            batch.isna().sum(),
            fill_value=0
        )

    del batch

print("\nОбщее количество строк:", f"{total_rows:,}")
print("\nРаспределение target:")
print(target_counts.sort_index().astype(int))

print("\nПропуски:")
print(missing_counts.astype(int))

Количество батчей: 20

Общее количество строк: 10,000,000

Распределение target:
target
0    9797694
1     167639
2      17063
3      17604
dtype: int64

Пропуски:
item_click_count         0
item_cart_count          0
item_order_count         0
item_total_count         0
item_recency             0
item_interaction_span    0
covisitation_score       0
item_click_rate          0
item_cart_rate           0
item_order_rate          0
session                  0
aid                      0
target                   0
dtype: int64


## Обучение модели на увеличенной выборке

После проверки датасета переходим к масштабированию модели ранжирования.

In [36]:
train_files = batch_files[:10]

train_parts = []

for file_path in train_files:
    part = pd.read_pickle(file_path)
    train_parts.append(part)

large_df = pd.concat(train_parts, ignore_index=True)

del train_parts
gc.collect()

print("Размер датасета:", large_df.shape)
print("Количество сессий:", large_df['session'].nunique())
print()
print(large_df['target'].value_counts().sort_index())

Размер датасета: (5000000, 13)
Количество сессий: 50000

target
0    4898598
1      84188
2       8530
3       8684
Name: count, dtype: int64


### Разделение на train и validation

Разделим данные на обучающую и валидационную выборки по `session`.

In [37]:
from sklearn.model_selection import train_test_split

sessions = large_df['session'].unique()

train_sessions, valid_sessions = train_test_split(
    sessions,
    test_size=0.2,
    random_state=55
)

train_data = large_df[large_df['session'].isin(train_sessions)].copy()

valid_data = large_df[large_df['session'].isin(valid_sessions)].copy()

print("Train:", train_data.shape)
print("Validation:", valid_data.shape)

print("Train sessions:", train_data['session'].nunique())
print("Validation sessions:", valid_data['session'].nunique())

Train: (4000000, 13)
Validation: (1000000, 13)
Train sessions: 40000
Validation sessions: 10000


### Подготовка данных для CatBoostRanker

Для обучения модели выделим признаки и целевую переменную.

In [40]:
from catboost import Pool

feature_columns = [
    'item_click_count',
    'item_cart_count',
    'item_order_count',
    'item_total_count',
    'item_recency',
    'item_interaction_span',
    'covisitation_score',
    'item_click_rate',
    'item_cart_rate',
    'item_order_rate'
]

train_data = train_data.sort_values('session').reset_index(drop=True)
valid_data = valid_data.sort_values('session').reset_index(drop=True)

train_pool = Pool(
    data=train_data[feature_columns],
    label=train_data['target'],
    group_id=train_data['session']
)

valid_pool = Pool(
    data=valid_data[feature_columns],
    label=valid_data['target'],
    group_id=valid_data['session']
)

print("Train Pool:", train_pool.num_row())
print("Validation Pool:", valid_pool.num_row())
print("Количество признаков:", train_pool.num_col())

Train Pool: 4000000
Validation Pool: 1000000
Количество признаков: 10


### Обучение CatBoostRanker

Обучим модель ранжирования на увеличенной выборке из 50 000 сессий.

В качестве функции потерь используем `YetiRank`, которая оптимизирует порядок объектов внутри каждой сессии.

In [41]:
from catboost import CatBoostRanker
import time

model = CatBoostRanker(
    loss_function='YetiRank',
    eval_metric='NDCG:top=20',
    iterations=500,
    learning_rate=0.05,
    depth=8,
    random_seed=55,
    verbose=50
)

start_time = time.time()

model.fit(
    train_pool,
    eval_set=valid_pool,
    early_stopping_rounds=50
)

training_time = time.time() - start_time

print(f"Время обучения: {training_time:.2f} сек.")

0:	test: 0.6887155	best: 0.6887155 (0)	total: 755ms	remaining: 6m 16s
50:	test: 0.7236844	best: 0.7237861 (47)	total: 31.5s	remaining: 4m 37s
100:	test: 0.7246907	best: 0.7249228 (79)	total: 1m 3s	remaining: 4m 11s
150:	test: 0.7248854	best: 0.7252014 (134)	total: 1m 36s	remaining: 3m 43s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.7252013953
bestIteration = 134

Shrink model to first 135 iterations.
Время обучения: 119.97 сек.


### Предсказания на validation

Получим score модели для каждого товара-кандидата. 

In [42]:
start_time = time.time()

valid_data['score'] = model.predict(valid_pool)

prediction_time = time.time() - start_time

print(f"Время предсказания: {prediction_time:.2f} сек.")
valid_data[['session', 'aid', 'target', 'score']].head(10)

Время предсказания: 0.07 сек.


,session,aid,target,score
0,0,661144,0,5.047819
1,0,1090479,0,0.375715
2,0,1164387,0,0.407222
3,0,1308544,0,0.863821
4,0,719622,0,1.016346
5,0,1443747,0,1.210114
6,0,1750538,0,1.863775
7,0,337364,0,1.246326
8,0,1653945,0,0.714371
9,0,1222638,0,1.196141


### Формирование Top-20 рекомендаций

Для каждой сессии отсортируем товары-кандидаты по предсказанному `score` и оставим 20 товаров, которые получили наибольшую оценку модели.

In [44]:
top20 = (
    valid_data
    .sort_values(['session', 'score'], ascending=[True, False])
    .groupby('session')
    .head(20)
)

print("Количество строк:", len(top20))
print("Количество сессий:", top20['session'].nunique())

top20[['session', 'aid', 'target', 'score']].head(20)

Количество строк: 200000
Количество сессий: 10000


,session,aid,target,score
0,0,661144,0,5.047819
69,0,275288,1,4.839775
70,0,974651,1,4.780827
71,0,1760145,1,4.667623
72,0,570955,0,4.641525
66,0,1157411,0,4.080632
64,0,102416,0,3.788202
61,0,1624436,0,3.536361
74,0,1436439,0,3.513208
68,0,543308,3,3.497668


### Оценка NDCG@20

Оценим качество итогового ранжирования на validation-выборке.

In [45]:
import numpy as np

def ndcg_at_20(group):
    relevance = group['target'].to_numpy()

    dcg = 0
    for i, rel in enumerate(relevance):
        dcg += (2 ** rel - 1) / np.log2(i + 2)

    ideal_relevance = np.sort(relevance)[::-1]

    idcg = 0
    for i, rel in enumerate(ideal_relevance):
        idcg += (2 ** rel - 1) / np.log2(i + 2)
    if idcg == 0:
        return np.nan

    return dcg / idcg


ndcg_values = (
    top20
    .groupby('session')
    .apply(ndcg_at_20)
    .dropna()
)

print("Количество сессий с релевантными товарами:", len(ndcg_values))
print(f"NDCG@20: {ndcg_values.mean():.4f}")

Количество сессий с релевантными товарами: 6421
NDCG@20: 0.7235


### NDCG@20 на полном наборе кандидатов

Для корректной оценки `NDCG@20` идеальный порядок строится по всем кандидатам сессии, а фактический `DCG` — по первым 20 товарам, отсортированным моделью.

Таким образом, метрика штрафует модель, если релевантный товар присутствовал среди кандидатов, но не попал в итоговый топ-20.

In [46]:
import numpy as np

def ndcg_at_k(group, k=20):
    ranked_relevance = (
        group
        .sort_values('score', ascending=False)
        ['target']
        .to_numpy()[:k]
    )

    dcg = 0
    for i, rel in enumerate(ranked_relevance):
        dcg += (2 ** rel - 1) / np.log2(i + 2)
    ideal_relevance = (
        np.sort(group['target'].to_numpy())[::-1][:k]
    )
    
    idcg = 0
    for i, rel in enumerate(ideal_relevance):
        idcg += (2 ** rel - 1) / np.log2(i + 2)
    if idcg == 0:
        return np.nan

    return dcg / idcg

ndcg_values = (valid_data.groupby('session').apply(ndcg_at_k).dropna())

print("Количество сессий с релевантными кандидатами:", len(ndcg_values))
print(f"NDCG@20: {ndcg_values.mean():.4f}")

Количество сессий с релевантными кандидатами: 6931
NDCG@20: 0.6010


Здесь используется valid_data со 100 кандидатами, а не только с товарами из топ-20.

## Результат эксперимента по масштабированию выборки

После увеличения обучающей выборки в 5 раз (до 50000 сессий) модель была повторна обучена и оценена на валидационной выборке.

Далее стоит сравнить качество модели на меньшей обучающей выборке, посмотреть важность признаков и сформулировать вывод по всему этапу скейлинга выборки.

### Контрольный эксперимент на меньшей обучающей выборке

Чтобы корректно оценить влияние объёма обучающих данных, обучим вторую модель на 8 000 сессиях.

In [49]:
small_train_sessions = train_sessions[:8000]

small_train_data = (
    train_data[train_data['session'].isin(small_train_sessions)]
    .copy()
    .sort_values('session')
    .reset_index(drop=True)
)

print("Размер:", small_train_data.shape)
print("Количество сессий:", small_train_data['session'].nunique())

Размер: (800000, 13)
Количество сессий: 8000


In [50]:
small_train_pool = Pool(
    data=small_train_data[feature_columns],
    label=small_train_data['target'],
    group_id=small_train_data['session']
)

print("Small Train Pool:", small_train_pool.num_row())

Small Train Pool: 800000


Используем тот же valid_pool, что и в большой модели, чтобы честно сравнить влияние размера обучающей выборки.

In [51]:
small_model = CatBoostRanker(
    loss_function='YetiRank',
    eval_metric='NDCG:top=20',
    iterations=500,
    learning_rate=0.05,
    depth=8,
    random_seed=55,
    verbose=50
)

start_time = time.time()

small_model.fit(
    small_train_pool,
    eval_set=valid_pool,
    early_stopping_rounds=50
)

small_training_time = time.time() - start_time

print(f"Время обучения: {small_training_time:.2f} сек.")

0:	test: 0.6933873	best: 0.6933873 (0)	total: 168ms	remaining: 1m 23s
50:	test: 0.7236621	best: 0.7236973 (49)	total: 8.31s	remaining: 1m 13s
100:	test: 0.7238729	best: 0.7242017 (88)	total: 16.6s	remaining: 1m 5s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.7242016827
bestIteration = 88

Shrink model to first 89 iterations.
Время обучения: 22.87 сек.


In [55]:
small_valid_data = valid_data.copy()

small_valid_data['score'] = small_model.predict(valid_pool)

small_ndcg_values = (small_valid_data.groupby('session').apply(ndcg_at_k).dropna())

small_model_ndcg = small_ndcg_values.mean()
print("Количество сессий с релевантными кандидатами:", len(small_ndcg_values))
print(f"NDCG@20 маленькой модели: {small_model_ndcg:.4f}")

Количество сессий с релевантными кандидатами: 6931
NDCG@20 маленькой модели: 0.5995


### Сравнение качества моделей

In [56]:
large_valid_data = valid_data.copy()

large_valid_data['score'] = model.predict(valid_pool)

large_ndcg_values = (large_valid_data.groupby('session').apply(ndcg_at_k).dropna())

large_model_ndcg = large_ndcg_values.mean()

print("Количество сессий с релевантными кандидатами:", len(large_ndcg_values))
print(f"NDCG@20 маленькой модели: {small_model_ndcg:.4f}")
print(f"NDCG@20 большой модели: {large_model_ndcg:.4f}")
print(f"Разница: {large_model_ndcg - small_model_ndcg:+.4f}")

Количество сессий с релевантными кандидатами: 6931
NDCG@20 маленькой модели: 0.5995
NDCG@20 большой модели: 0.6010
Разница: +0.0015


### Качество и стоимость масштабирования

Увеличение обучающей выборки позволяет проверить, насколько качество модели зависит от количества данных. Сравним изменение **NDCG@20** и времени обучения маленькой и большой моделей.

In [57]:
comparison = pd.DataFrame({
    'model': ['10k sessions', '40k sessions'],
    'NDCG@20': [small_model_ndcg, large_model_ndcg],
    'training_time_sec': [small_training_time, training_time]
})
comparison

,model,NDCG@20,training_time_sec
0,10k sessions,0.599497,22.867802
1,40k sessions,0.601020,119.974532


## Важность признаков

После обучения модели на увеличенной выборке проанализируем важность признаков.

In [59]:
feature_importance = pd.DataFrame({
    'feature': feature_columns,
    'importance': model.get_feature_importance(train_pool)
}).sort_values('importance', ascending=False)

feature_importance

,feature,importance
4,item_recency,0.107245
5,item_interaction_span,0.033164
6,covisitation_score,0.007290
9,item_order_rate,0.000078
1,item_cart_count,0.000054
2,item_order_count,0.000049
3,item_total_count,-0.000129
8,item_cart_rate,-0.000179
0,item_click_count,-0.000191
7,item_click_rate,-0.000228


## Итоги эксперимента

Основные результаты:

- датасет был увеличен до 50 000 сессий и 5 млн пар `session-item`;
- модель обучалась на 40 000 сессий, а 10 000 использовались для валидации;
- `NDCG@20` модели, обученной на 10 000 сессий, составил 0.5995;
- `NDCG@20` модели, обученной на 40 000 сессий, составил 0.6010;
- увеличение обучающей выборки в 4 раза дало небольшой прирост NDCG@20 — около 0.0015;
- время обучения при этом выросло с 22.9 до 120 секунд;
- наиболее полезными признаками оказались `item_recency`, `item_interaction_span` и `covisitation_score`.

Выяснилось, что увеличание обучающей выборки не приводит к сопоставимому росту качества, но зато заметно повышает время обучения.